# Análisis Exploratorio de Datos (EDA) y Limpieza del Dataset SPARCS
**Curso:** Programación Concurrente y Distribuida (CC65)  
**Caso de Estudio:** Optimización de costos y recursos hospitalarios (ODS 3: Salud y bienestar)  
**Fuente:** Health Data NY - *Hospital Inpatient Discharges (SPARCS De-Identified)* (>2.1 millones de registros)  

---

## 1. Importación de Librerías y Configuración del Entorno
Importamos las librerías necesarias para el análisis tabular y gráfico.

In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Configuración de visualización
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
pd.set_option('display.float_format', lambda x: '%.2f' % x)
pd.set_option('display.max_columns', None)

DATASET_PATH = os.path.join('dataset', 'Hospital_Inpatient_Discharges_(SPARCS_De-Identified)__2022_20260910.csv')
print('Ruta del dataset:', DATASET_PATH)
print('¿El archivo existe?:', os.path.exists(DATASET_PATH))

## 2. Inspección Inicial del Archivo Crudo
Dado que el archivo supera los **900 MB**, primero inspeccionamos las primeras filas y los nombres de las 33 columnas originales sin sobrecargar la memoria.

In [ ]:
# Lectura de muestra inicial (primeras 5 filas)
sample_df = pd.read_csv(DATASET_PATH, nrows=5)
print(f'Total de columnas en el dataset original: {sample_df.shape[1]}')
print('\nListado de columnas:')
for idx, col in enumerate(sample_df.columns, 1):
    print(f'{idx}. {col}')

sample_df.head(3)

## 3. Selección de Columnas Relevantes (Feature Selection)
De las 33 columnas disponibles, seleccionamos únicamente las 8 variables requeridas para el modelo de predicción de costos hospitalarios y análisis de estancia:
1. `Permanent Facility Id`: Identificador del hospital.
2. `Age Group`: Grupo etario del paciente.
3. `Length of Stay`: Días de estancia hospitalaria (predictora clave).
4. `APR DRG Description`: Diagnóstico y procedimiento clínico.
5. `Payment Typology 1`: Tipo de pagador / asegurador.
6. `Emergency Department Indicator`: Si el ingreso fue por urgencias.
7. `Total Charges`: Cargos totales facturados.
8. `Total Costs`: Costo total real de la atención (variable objetivo).

In [ ]:
selected_columns = [
    'Permanent Facility Id',
    'Age Group',
    'Length of Stay',
    'APR DRG Description',
    'Payment Typology 1',
    'Emergency Department Indicator',
    'Total Charges',
    'Total Costs'
]

# Carga de una muestra de 200,000 registros para el análisis exploratorio detallado
print('Cargando muestra de 200,000 registros para EDA...')
df_eda = pd.read_csv(DATASET_PATH, nrows=200000, usecols=selected_columns, low_memory=False)
print(f'Dimensiones de la muestra: {df_eda.shape}')
df_eda.info()

## 4. Diagnóstico de Anomalías en los Datos
En esta sección detectamos:
1. **Formatos monetarios:** `Total Charges` y `Total Costs` vienen como `object` (texto) debido al uso de comas como separador de miles.
2. **Codificación especial en estancia:** `Length of Stay` incluye valores como `'120 +'` para estancias prolongadas.
3. **Valores nulos o vacíos:** Identificamos qué columnas presentan valores faltantes.

In [ ]:
print('Valores nulos por columna:')
print(df_eda.isna().sum())

print('\nEjemplos de Total Charges con comas:')
print(df_eda['Total Charges'].dropna().head(5).tolist())

print('\nCasos no numéricos en Length of Stay (ej. 120 +):')
print(df_eda[df_eda['Length of Stay'].astype(str).str.contains(r'\+', regex=True)]['Length of Stay'].value_counts())

## 5. Implementación del Procedimiento de Limpieza y Estandarización
Definimos funciones de transformación para:
- Quitar comas y símbolos de moneda, convirtiendo a numérico flotante (`float64`).
- Estandarizar `'120 +'` a `120` entero.
- Imputar categorías faltantes con `'No especificado'`.
- Descartar registros con montos o estancias menores o iguales a cero.

In [ ]:
def clean_currency(series):
    cleaned = series.astype(str).str.replace(r'[$, ]', '', regex=True)
    return pd.to_numeric(cleaned, errors='coerce')

def clean_stay(series):
    cleaned = series.astype(str).str.strip().str.replace(r'^120\s*\+.*', '120', regex=True)
    return pd.to_numeric(cleaned, errors='coerce')

# Aplicación de transformaciones
df_clean = df_eda.copy()
df_clean['Total Charges'] = clean_currency(df_clean['Total Charges'])
df_clean['Total Costs'] = clean_currency(df_clean['Total Costs'])
df_clean['Length of Stay'] = clean_stay(df_clean['Length of Stay'])

# Imputación de variables categóricas
df_clean['Payment Typology 1'] = df_clean['Payment Typology 1'].fillna('No especificado')
df_clean['APR DRG Description'] = df_clean['APR DRG Description'].fillna('No especificado')

# Filtrado de nulos en variables numéricas clave
df_clean = df_clean.dropna(subset=['Total Costs', 'Total Charges', 'Length of Stay'])

# Filtrado de inconsistencias de negocio (costos y días positivos)
df_clean = df_clean[(df_clean['Total Costs'] > 0) & (df_clean['Total Charges'] > 0) & (df_clean['Length of Stay'] > 0)]

print('Dimensiones tras limpieza básica:', df_clean.shape)
df_clean.dtypes

## 6. Detección y Tratamiento de Outliers (Rango Intercuartílico - IQR)
Analizamos la distribución de `Total Costs` y `Length of Stay` para identificar valores atípicos mediante el método de Tukey (IQR):
$$IQR = Q_3 - Q_1$$
$$[Límite_{inf}, Límite_{sup}] = [Q_1 - 1.5 \times IQR, \, Q_3 + 1.5 \times IQR]$$

In [ ]:
# Cálculo de IQR para Total Costs
q1_cost = df_clean['Total Costs'].quantile(0.25)
q3_cost = df_clean['Total Costs'].quantile(0.75)
iqr_cost = q3_cost - q1_cost
upper_bound_cost = q3_cost + 1.5 * iqr_cost
lower_bound_cost = max(0, q1_cost - 1.5 * iqr_cost)

# Cálculo de IQR para Length of Stay
q1_los = df_clean['Length of Stay'].quantile(0.25)
q3_los = df_clean['Length of Stay'].quantile(0.75)
iqr_los = q3_los - q1_los
upper_bound_los = q3_los + 1.5 * iqr_los

print(f'Total Costs: Q1={q1_cost:.2f}, Q3={q3_cost:.2f}, IQR={iqr_cost:.2f}')
print(f'Límite superior para Total Costs: ${upper_bound_cost:.2f}')
print(f'Límite superior para Length of Stay: {upper_bound_los:.1f} días')

# Visualización de Boxplots antes del filtrado
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.boxplot(y=df_clean['Total Costs'], ax=axes[0], color='skyblue')
axes[0].set_title('Distribución de Total Costs (con outliers)')
axes[0].set_ylabel('Costo Total ($)')

sns.boxplot(y=df_clean['Length of Stay'], ax=axes[1], color='salmon')
axes[1].set_title('Distribución de Length of Stay (con outliers)')
axes[1].set_ylabel('Días de Estancia')
plt.tight_layout()
plt.show()

## 7. Filtrado de Outliers y Comparación de Distribuciones

In [ ]:
# Aplicar filtro de outliers
df_filtered = df_clean[
    (df_clean['Total Costs'] <= upper_bound_cost) &
    (df_clean['Length of Stay'] <= upper_bound_los)
].copy()

print(f'Registros antes de remover outliers: {len(df_clean)}')
print(f'Registros después de remover outliers: {len(df_filtered)}')
print(f'Outliers removidos: {len(df_clean) - len(df_filtered)} ({(len(df_clean) - len(df_filtered)) / len(df_clean) * 100:.2f}%)')

# Visualización comparativa
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.histplot(df_filtered['Total Costs'], kde=True, ax=axes[0], color='teal', bins=30)
axes[0].set_title('Distribución de Total Costs Depurado')
axes[0].set_xlabel('Costo Total ($)')

sns.histplot(df_filtered['Length of Stay'], kde=False, ax=axes[1], color='coral', discrete=True)
axes[1].set_title('Distribución de Length of Stay Depurado')
axes[1].set_xlabel('Días de Estancia')
plt.tight_layout()
plt.show()

## 8. Análisis de Correlación y Estadísticas Descriptivas Finales
Generamos la tabla resumen de estadísticas y la correlación entre variables numéricas para sustentar el modelo de Regresión Lineal.

In [ ]:
print('Estadísticas descriptivas finales:')
display(df_filtered[['Length of Stay', 'Total Charges', 'Total Costs']].describe())

# Matriz de correlación
plt.figure(figsize=(7, 5))
corr = df_filtered[['Length of Stay', 'Total Charges', 'Total Costs']].corr()
sns.heatmap(corr, annot=True, cmap='Blues', fmt='.2f', vmin=0, vmax=1)
plt.title('Matriz de Correlación (Variables Numéricas)')
plt.show()

## 9. Procesamiento del Dataset Masivo Completo (>2.1M registros)
Una vez validadas las reglas en la muestra, procesamos el archivo completo por bloques (chunks) para generar el dataset limpio final.

In [ ]:
OUTPUT_CLEAN_PATH = os.path.join('dataset', 'SPARCS_2022_clean.csv')
chunk_size = 100000

total_read = 0
total_clean = 0
first_chunk = True

print('Procesando dataset completo en chunks de 100,000 filas...')
for chunk in pd.read_csv(DATASET_PATH, chunksize=chunk_size, usecols=selected_columns, low_memory=False):
    total_read += len(chunk)
    
    # Limpieza de tipos
    chunk['Total Charges'] = clean_currency(chunk['Total Charges'])
    chunk['Total Costs'] = clean_currency(chunk['Total Costs'])
    chunk['Length of Stay'] = clean_stay(chunk['Length of Stay'])
    
    chunk['Payment Typology 1'] = chunk['Payment Typology 1'].fillna('No especificado')
    chunk['APR DRG Description'] = chunk['APR DRG Description'].fillna('No especificado')
    
    # Filtrado de nulos y consistencia
    c_clean = chunk.dropna(subset=['Total Costs', 'Total Charges', 'Length of Stay'])
    c_clean = c_clean[(c_clean['Total Costs'] > 0) & (c_clean['Total Charges'] > 0) & (c_clean['Length of Stay'] > 0)]
    
    # Filtrado de outliers extremos
    c_clean = c_clean[(c_clean['Length of Stay'] <= 365) & (c_clean['Total Costs'] <= 1000000)]
    
    total_clean += len(c_clean)
    
    # Guardado incremental
    c_clean.to_csv(OUTPUT_CLEAN_PATH, mode='w' if first_chunk else 'a', index=False, header=first_chunk)
    first_chunk = False

print(f'Proceso completado exitosamente.')
print(f'Total de filas leídas: {total_read:,}')
print(f'Total de filas limpias guardadas: {total_clean:,}')
print(f'Tasa de retención: {total_clean / total_read * 100:.2f}%')
print(f'Archivo guardado en: {OUTPUT_CLEAN_PATH}')